In [7]:
#!/usr/bin/env python3
"""
XLM-RoBERTa-large + CRF with hard BIO transition constraints
Optimized Hyperparameters for Contextual Learning & Regularization.
INCLUDES LIVE DIAGNOSTICS & ANOMALY TRACKING.
"""

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import sys
import gc
import json
import torch
import subprocess
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm
from datasets import Dataset, load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    DataCollatorForTokenClassification,
    get_linear_schedule_with_warmup
)
from torchcrf import CRF
import shutil

# ------------------------------------------------------------------
# Optimized Hyperparameters
# ------------------------------------------------------------------
MODEL_PATH      = "xlm-roberta-large"
OUTPUT_DIR      = "./xlm_roberta_large_crfV2"
CHECKPOINT_DIR  = "./xlm_roberta_large_checkpointV2"
DATA_CACHE_DIR  = "./xlm_roberta_large_cacheV2"
DIAGNOSTIC_LOG  = "training_diagnostics.log"  # <--- NEW LOG FILE

TRAIN_FILE = "data/train.jsonl"
VAL_FILE   = "data/validation.jsonl"
TEST_FILE  = "data/test.jsonl"

EPOCHS             = 5         
LEARNING_RATE      = 8e-6      
WEIGHT_DECAY       = 0.05      
BATCH_SIZE         = 16        
ACCUMULATION_STEPS = 8         
MAX_LEN            = 128
SAVE_STEPS         = 300
# ------------------------------------------------------------------

class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(
            model_name_or_path, config=config, ignore_mismatched_sizes=True
        )

        self.dropout = torch.nn.Dropout(0.2) 
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)

        self.crf = CRF(num_tags=config.num_labels, batch_first=True)
        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        num_tags = len(label_list)
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, :] = 0.0
        self.crf.transitions.data[:, o_id] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O":
                continue
            tid = label2id[tag]
            self.crf.start_transitions.data[tid] = 0.0
            self.crf.end_transitions.data[tid] = 0.0

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.bool()

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            loss = -self.crf(emissions, tags=safe_labels, mask=mask, reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=mask)

    def save_pretrained(self, save_directory):
        os.makedirs(save_directory, exist_ok=True)
        self.config.save_pretrained(save_directory)
        torch.save(self.state_dict(), os.path.join(save_directory, "pytorch_model.bin"))

def get_emptiest_gpu_safely():
    if not torch.cuda.is_available(): return torch.device("cpu")
    try:
        print("\n🔍 Scanning available GPUs safely via nvidia-smi...")
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free,utilization.gpu", "--format=csv,nounits,noheader"],
            encoding="utf-8"
        )
        best_id, max_free_mb = -1, 0
        fallback_id, fallback_max_mb = 0, 0

        for line in result.strip().split("\n"):
            parts = line.split(", ")
            gpu_id = int(parts[0])
            free_memory = int(parts[1])
            gpu_util = int(parts[2])

            if free_memory > fallback_max_mb:
                fallback_max_mb = free_memory
                fallback_id = gpu_id

            if gpu_util < 30 and free_memory > max_free_mb:
                max_free_mb = free_memory
                best_id = gpu_id

        if best_id != -1:
            print(f"--> Selected GPU {best_id} with {max_free_mb} MB free VRAM.\n")
            return torch.device(f"cuda:{best_id}")
        else:
            return torch.device(f"cuda:{fallback_id}")
    except Exception: return torch.device("cuda:0")

def reconstruct_char_labels(input_text, output_dict):
    char_labels = ["O"] * len(input_text)

    field_to_tag = {
        "flat": "UNIT",
        "floor": "FLOOR",
        "block": "BLOCK",
        "phase": "PHASE",
        "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME",
        "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT",
        "district": "DISTRICT",
        "region": "REGION",
        "village_name": "VILLAGE_NAME",
        "building_number": "BUILDING_NUMBER"
    }

    flat_targets = {}
    if "line1" in output_dict:
        flat_targets.update(output_dict.get("line1", {}))
        flat_targets.update(output_dict.get("line2", {}))
    else:
        flat_targets = output_dict

    items_to_process = []
    for field, tag in field_to_tag.items():
        val = flat_targets.get(field, "")
        if not val:
            continue
        search_terms = val.split(" / ") if " / " in val else [val]
        for term in search_terms:
            if term:
                items_to_process.append((field, tag, term))

    # Longer matches first
    items_to_process.sort(key=lambda x: len(x[2]), reverse=True)

    # Convert the main text to lowercase once for searching
    lower_input = input_text.lower()

    for field, tag, term in items_to_process:
        start_idx = 0
        lower_term = term.lower() # Convert search term to lowercase
        
        while True:
            # Search using the lowercase versions
            idx = lower_input.find(lower_term, start_idx)
            if idx == -1:
                break
                
            is_already_tagged = any(
                char_labels[i] != "O" for i in range(idx, idx + len(term))
            )
            
            if not is_already_tagged:
                char_labels[idx] = f"B-{tag}"
                for i in range(idx + 1, idx + len(term)):
                    if i < len(char_labels):
                        char_labels[i] = f"I-{tag}"
                break
            else:
                start_idx = idx + 1

    return char_labels

def parse_and_tokenize(tokenizer):
    print("📂 Parsing datasets and reconstructing NER labels...")
    unique_labels = {"O"}
    tag_list = [
        "UNIT", "FLOOR", "BUILDING_NAME", "ESTATE_NAME", "STREET_NAME",
        "SUB_DISTRICT", "DISTRICT", "REGION", "VILLAGE_NAME", "BUILDING_NUMBER",
        "BLOCK", "PHASE"
    ]
    for tag in tag_list:
        unique_labels.add(f"B-{tag}")
        unique_labels.add(f"I-{tag}")

    label_list = sorted(list(unique_labels))
    label_to_id = {l: i for i, l in enumerate(label_list)}

    datasets_dict = {}
    for file_path in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
        if not os.path.exists(file_path): continue

        texts, labels_list = [], []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in tqdm(f, desc=f"Reading {file_path}"):
                line = line.strip()
                if not line: continue
                data = json.loads(line)
                input_text = data.get("input", "")
                output_dict = data.get("output", {})
                char_labels = reconstruct_char_labels(input_text, output_dict)
                texts.append(input_text)
                labels_list.append(char_labels)

        datasets_dict[file_path] = Dataset.from_dict({"text": texts, "char_labels": labels_list})

    os.makedirs(DATA_CACHE_DIR, exist_ok=True)
    with open(os.path.join(DATA_CACHE_DIR, "label_map.json"), "w") as f:
        json.dump({"label_list": label_list, "label_to_id": label_to_id}, f)

    def align_labels(examples):
        tokenized = tokenizer(
            examples["text"], truncation=True, max_length=MAX_LEN, return_offsets_mapping=True
        )
        labels = []
        for i, offsets in enumerate(tokenized["offset_mapping"]):
            char_labels = examples["char_labels"][i]
            token_labels = []
            for start_offset, end_offset in offsets:
                if start_offset == end_offset:
                    token_labels.append(-100)
                else:
                    tag = "O"
                    for char_idx in range(start_offset, end_offset):
                        if char_idx < len(char_labels) and char_labels[char_idx] != "O":
                            tag = char_labels[char_idx]
                            break
                    token_labels.append(label_to_id[tag])
            labels.append(token_labels)

        tokenized["labels"] = labels
        tokenized.pop("offset_mapping")
        return tokenized

    print("⏳ Tokenizing datasets...")
    train_ds = datasets_dict[TRAIN_FILE].map(align_labels, batched=True, remove_columns=["text", "char_labels"])
    val_ds = datasets_dict.get(VAL_FILE)
    if val_ds:
        val_ds = val_ds.map(align_labels, batched=True, remove_columns=["text", "char_labels"])

    print(f"💾 Caching tokenized datasets to {DATA_CACHE_DIR}...")
    train_ds.save_to_disk(os.path.join(DATA_CACHE_DIR, "train"))
    if val_ds: val_ds.save_to_disk(os.path.join(DATA_CACHE_DIR, "val"))

    return train_ds, val_ds, label_list, label_to_id

def save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, best_val_loss):
    print(f"\n💾 Saving checkpoint atomically at Epoch {epoch+1}, Step {step}...")
    temp_dir = f"{CHECKPOINT_DIR}_tmp"
    os.makedirs(temp_dir, exist_ok=True)
    model.save_pretrained(temp_dir)
    tokenizer.save_pretrained(temp_dir)
    state = {
        "epoch": epoch, "step": step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_loss": best_val_loss
    }
    torch.save(state, os.path.join(temp_dir, "training_state.pt"))
    try: os.replace(temp_dir, CHECKPOINT_DIR)
    except OSError:
        if os.path.exists(CHECKPOINT_DIR): shutil.rmtree(CHECKPOINT_DIR)
        os.rename(temp_dir, CHECKPOINT_DIR)

# --- NEW DIAGNOSTIC HELPER FUNCTION ---
def log_diagnostic(tokenizer, input_ids, labels, preds, id2label, metric_val, epoch, step, context):
    """Silently logs an anomaly or mismatch to a file for later inspection."""
    with open(DIAGNOSTIC_LOG, "a", encoding="utf-8") as f:
        f.write(f"\n[{context}] Epoch {epoch+1} | Step {step} | Metric/Loss: {metric_val:.4f}\n")
        
        tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
        true_ids = labels[0].tolist()
        pred_ids = preds[0]
        
        f.write(f"{'TOKEN':<20} | {'TRUE LABEL':<20} | {'PRED LABEL':<20}\n")
        f.write("-" * 65 + "\n")
        
        pred_idx = 0
        for tok, t_id in zip(tokens, true_ids):
            if tok in [tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token]: continue
            if t_id == -100: continue
            
            # Robust mapping
            true_tag = id2label[t_id] if isinstance(t_id, int) else id2label.get(str(t_id), "O")
            
            p_id = pred_ids[pred_idx] if pred_idx < len(pred_ids) else 0
            pred_tag = id2label[p_id] if isinstance(p_id, int) else id2label.get(str(p_id), "O")
            
            pred_idx += 1
            marker = "❌" if true_tag != pred_tag else "✅"
            f.write(f"{tok:<20} | {true_tag:<20} | {pred_tag:<20} {marker}\n")
        f.write("=" * 65 + "\n")


def main():
    # Initialize the silent log file
    with open(DIAGNOSTIC_LOG, "w", encoding="utf-8") as f:
        f.write("🚀 Starting New Diagnostic Log Run\n" + "="*40 + "\n")

    device = get_emptiest_gpu_safely()
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

    if os.path.exists(os.path.join(DATA_CACHE_DIR, "train")):
        train_ds = load_from_disk(os.path.join(DATA_CACHE_DIR, "train"))
        val_ds = load_from_disk(os.path.join(DATA_CACHE_DIR, "val"))
        with open(os.path.join(DATA_CACHE_DIR, "label_map.json"), "r") as f:
            label_data = json.load(f)
            label_list = label_data["label_list"]
            label_to_id = {k: int(v) for k, v in label_data["label_to_id"].items()}
    else:
        train_ds, val_ds, label_list, label_to_id = parse_and_tokenize(tokenizer)

    # Reverse lookup for diagnostics
    id2label = {int(v): k for k, v in label_to_id.items()}

    print("⏳ Initialising XLM-RoBERTa-large + constrained CRF...")
    resume_from_checkpoint = os.path.exists(os.path.join(CHECKPOINT_DIR, "training_state.pt"))
    config = AutoConfig.from_pretrained(MODEL_PATH, num_labels=len(label_list), id2label=id2label, label2id=label_to_id)
    model = BertCRFForTokenClassification(config, MODEL_PATH, label_list)

    if resume_from_checkpoint:
        weights_path = os.path.join(CHECKPOINT_DIR, "pytorch_model.bin")
        if os.path.exists(weights_path):
            print("🔄 Loading weights from checkpoint...")
            model.load_state_dict(torch.load(weights_path, map_location=device))
            model._set_bio_constraints(label_list)

    model.to(device)
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    train_loader = DataLoader(train_ds, shuffle=True, batch_size=BATCH_SIZE, collate_fn=data_collator, num_workers=4, pin_memory=True)
    val_loader = None
    if val_ds:
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, collate_fn=data_collator, num_workers=4, pin_memory=True)

    torch.cuda.empty_cache()
    gc.collect()

    optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = (len(train_loader) // ACCUMULATION_STEPS) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.2), num_training_steps=total_steps)

    start_epoch = 0
    start_step = 0
    best_val_loss = float("inf")

    if resume_from_checkpoint:
        state = torch.load(os.path.join(CHECKPOINT_DIR, "training_state.pt"), map_location=device)
        optimizer.load_state_dict(state["optimizer"])
        scheduler.load_state_dict(state["scheduler"])
        start_epoch = state["epoch"]
        start_step = state["step"]
        best_val_loss = state.get("best_val_loss", float("inf"))
        print(f"⏩ Fast-forwarding to Epoch {start_epoch+1}, Step {start_step}...")

    print("\n🚀 Training XLM-RoBERTa-large + constrained CRF...")

    try:
        epoch_pbar = tqdm(range(start_epoch, EPOCHS), desc="Epochs", initial=start_epoch, total=EPOCHS)
        
        # Track a moving average to spot abnormal spikes
        running_loss = 0.0

        for epoch in epoch_pbar:
            model.train()
            total_train_loss = 0
            optimizer.zero_grad()

            batch_pbar = tqdm(train_loader, desc=f"Train (Ep {epoch+1})", leave=False)
            for step, batch in enumerate(batch_pbar):
                if epoch == start_epoch and step < start_step: continue

                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["labels"].to(device)

                loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                loss = loss / ACCUMULATION_STEPS
                loss.backward()

                if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                display_loss = loss.item() * ACCUMULATION_STEPS
                total_train_loss += display_loss
                
                # Setup moving average baseline
                if running_loss == 0.0: running_loss = display_loss
                else: running_loss = 0.9 * running_loss + 0.1 * display_loss

                # --- THE SPIKE CATCHER ---
                # If loss is > 30 (severe anomaly due to CRF punishment), capture it!
                if display_loss > 30.0:
                    model.eval()
                    with torch.no_grad(): # Do a quick prediction without labels to see what the model is thinking
                        spike_preds = model(input_ids=input_ids, attention_mask=attention_mask)
                    model.train()
                    log_diagnostic(tokenizer, input_ids, labels, spike_preds, id2label, display_loss, epoch, step, "TRAIN_LOSS_SPIKE")

                batch_pbar.set_postfix({"loss": f"{display_loss:.4f}"})

                if (step + 1) % SAVE_STEPS == 0:
                    save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step + 1, best_val_loss)

            avg_train_loss = total_train_loss / (len(train_loader) - (start_step if epoch == start_epoch else 0))
            start_step = 0

            # --- VALIDATION LOOP (NOW WITH EXACT MATCH TRACKING) ---
            if val_loader:
                model.eval()
                total_val_loss = 0
                total_exact_match = 0
                val_samples_processed = 0

                with torch.no_grad():
                    val_pbar = tqdm(val_loader, desc=f"Val (Ep {epoch+1})", leave=False)
                    for batch in val_pbar:
                        input_ids = batch["input_ids"].to(device)
                        attention_mask = batch["attention_mask"].to(device)
                        labels = batch["labels"].to(device)
                        
                        # 1. Get Loss
                        val_loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                        total_val_loss += val_loss.item()

                        # 2. Get Actual Predictions
                        val_preds = model(input_ids=input_ids, attention_mask=attention_mask)
                        
                        # 3. Calculate Sequence-Level Exact Match
                        mask = attention_mask.bool()
                        safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
                        
                        for i in range(len(val_preds)):
                            pred_seq = val_preds[i]
                            true_seq = safe_labels[i][mask[i]].tolist()
                            
                            if pred_seq == true_seq:
                                total_exact_match += 1
                            else:
                                # THE MISMATCH CATCHER: Log a few errors periodically so you can see why it's failing
                                if val_samples_processed % 400 == 0:
                                    log_diagnostic(tokenizer, input_ids[i:i+1], labels[i:i+1], [pred_seq], id2label, val_loss.item(), epoch, step, "VAL_MISMATCH_SAMPLE")
                            
                            val_samples_processed += 1

                avg_val_loss = total_val_loss / len(val_loader)
                val_em_percentage = total_exact_match / val_samples_processed

                # NOW SHOWS EXACT MATCH LIVE ON THE TERMINAL!
                epoch_pbar.set_postfix({
                    "Train Loss": f"{avg_train_loss:.4f}",
                    "Val Loss": f"{avg_val_loss:.4f}",
                    "Val EM": f"{val_em_percentage:.1%}" 
                })

                save_checkpoint(model, tokenizer, optimizer, scheduler, epoch + 1, 0, best_val_loss)

                if avg_val_loss < best_val_loss:
                    best_val_loss = avg_val_loss
                    print(f"\n🌟 New best val loss ({best_val_loss:.4f}) → {OUTPUT_DIR}")
                    os.makedirs(OUTPUT_DIR, exist_ok=True)
                    uncompiled = getattr(model, "_orig_mod", model)
                    uncompiled.save_pretrained(OUTPUT_DIR)
                    tokenizer.save_pretrained(OUTPUT_DIR)

        print(f"✅ Training complete. Best model saved to {OUTPUT_DIR}")

    except KeyboardInterrupt:
        print("\n\n⚠️ Training interrupted by user!")
        save_checkpoint(model, tokenizer, optimizer, scheduler, epoch, step, best_val_loss)
        print("👋 Safe exit. Re-run the script to resume.")
        sys.exit(0)

if __name__ == "__main__":
    main()


🔍 Scanning available GPUs safely via nvidia-smi...
--> Selected GPU 3 with 18054 MB free VRAM.

📂 Parsing datasets and reconstructing NER labels...


Reading data/train.jsonl: 0it [00:00, ?it/s]

Reading data/validation.jsonl: 0it [00:00, ?it/s]

Reading data/test.jsonl: 0it [00:00, ?it/s]

⏳ Tokenizing datasets...


Map:   0%|          | 0/165564 [00:00<?, ? examples/s]

KeyError: 'B-DISTRICT'

In [ ]:
#!/usr/bin/env python3
"""
Batch Evaluation Script for XLM-RoBERTa-large + Constrained CRF
Includes Multi-Threshold Confidence Diagnostics (Baseline, 80%, 85%, 90%, 95%).
"""

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import re
import json
import time
import torch
import subprocess
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoConfig, AutoModel
from torchcrf import CRF

# ==========================================
# CONFIGURATION
# ==========================================
MODEL_DIR = "./xlm_roberta_large_checkpointV2"
LOG_FILE = "parsing_results_xlm_largeV2.log"
TEST_FILE = "data/test.jsonl"
MAX_LEN = 128
BATCH_SIZE = 32

EXCLUDE_FROM_OVERALL = {"DISTRICT", "REGION", "SUB_DISTRICT"}
THRESHOLDS = [0.0, 0.80, 0.85, 0.90, 0.95] # 0.0 is the Baseline (All predictions)

# ==========================================
# CUSTOM ARCHITECTURE
# ==========================================
class BertCRFForTokenClassification(torch.nn.Module):
    def __init__(self, config, model_name_or_path, label_list):
        super().__init__()
        self.config = config
        self.label_list = label_list

        self.bert = AutoModel.from_pretrained(model_name_or_path, config=config, ignore_mismatched_sizes=True)
        self.dropout = torch.nn.Dropout(config.hidden_dropout_prob)
        self.classifier = torch.nn.Linear(config.hidden_size, config.num_labels)
        self.crf = CRF(num_tags=config.num_labels, batch_first=True)

        self._set_bio_constraints(label_list)

    def _set_bio_constraints(self, label_list):
        num_tags = len(label_list)
        self.crf.transitions.data.fill_(-1e4)
        self.crf.start_transitions.data.fill_(-1e4)
        self.crf.end_transitions.data.fill_(-1e4)

        id2label = {i: l for i, l in enumerate(label_list)}
        label2id = {l: i for i, l in id2label.items()}

        o_id = label2id["O"]
        self.crf.transitions.data[o_id, :] = 0.0
        self.crf.transitions.data[:, o_id] = 0.0
        self.crf.start_transitions.data[o_id] = 0.0
        self.crf.end_transitions.data[o_id] = 0.0

        for tag in label_list:
            if tag == "O": continue
            tid = label2id[tag]
            self.crf.start_transitions.data[tid] = 0.0
            self.crf.end_transitions.data[tid] = 0.0

            if tag.startswith("B-"):
                itype = "I-" + tag[2:]
                if itype in label2id:
                    self.crf.transitions.data[tid, label2id[itype]] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0
            elif tag.startswith("I-"):
                self.crf.transitions.data[tid, tid] = 0.0
                for other in label_list:
                    if other.startswith("B-"):
                        self.crf.transitions.data[tid, label2id[other]] = 0.0

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        sequence_output = self.dropout(outputs[0])
        emissions = self.classifier(sequence_output)
        mask = attention_mask.type(torch.uint8)

        if labels is not None:
            safe_labels = torch.where(labels >= 0, labels, torch.zeros_like(labels))
            return -self.crf(emissions, tags=safe_labels, mask=mask, reduction='mean')
        else:
            tags = self.crf.decode(emissions, mask=mask)
            return tags, emissions

# ==========================================
# UTILITY FUNCTIONS
# ==========================================
def get_emptiest_gpu_safely():
    if not torch.cuda.is_available(): return -1
    try:
        result = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,nounits,noheader"], 
            encoding="utf-8"
        )
        best_id, max_free_mb = 0, -1
        for line in result.strip().split("\n"):
            if not line.strip(): continue
            gpu_id, free_memory = map(int, line.split(", "))
            if free_memory > max_free_mb:
                max_free_mb, best_id = free_memory, gpu_id
        return best_id
    except Exception: return 0

def extract_3d_components(parsed_entities):
    components = defaultdict(list)
    confs = defaultdict(list)

    for entity in parsed_entities:
        tag = entity["entity_group"]
        if tag == "O": continue
        word = entity["word"].strip()
        if word:
            components[tag].append(word)
            confs[tag].append(entity["conf"])

    formatted_output, conf_output = {}, {}
    for tag, words in components.items():
        joined_string = "".join(words)
        if any("\u4e00" <= char <= "\u9fff" for char in joined_string):
            formatted_output[tag] = "".join(words)
        else:
            joined_en = " ".join(words).strip()
            formatted_output[tag] = re.sub(r"\s*([/\.-])\s*", r"\1", joined_en)

        conf_output[tag] = sum(confs[tag]) / len(confs[tag]) if confs[tag] else 0.0

    return formatted_output, conf_output

def assemble_compact_json(extracted_data):
    return {
        "line1": {
            "flat": extracted_data.get("UNIT", ""), "floor": extracted_data.get("FLOOR", ""),
            "block": extracted_data.get("BLOCK", ""), "phase": extracted_data.get("PHASE", ""),
            "building_name": extracted_data.get("BUILDING_NAME", ""),
        },
        "line2": {
            "estate_name": extracted_data.get("ESTATE_NAME", ""), "village_name": extracted_data.get("VILLAGE_NAME", ""),        
            "building_number": extracted_data.get("BUILDING_NUMBER", ""), "street_name": extracted_data.get("STREET_NAME", ""),
            "sub_district": extracted_data.get("SUB_DISTRICT", ""), "district": extracted_data.get("LOCATION", ""),
            "region": extracted_data.get("REGION", ""),
        }
    }

def flatten_json(output_dict):
    flat = {}
    if "line1" in output_dict:
        flat.update(output_dict.get("line1", {}))
        flat.update(output_dict.get("line2", {}))
    else:
        flat = output_dict
    return {k: v for k, v in flat.items() if v}

def compute_overall_confidence(conf_output):
    overall = 1.0
    for tag, c in conf_output.items():
        if tag not in EXCLUDE_FROM_OVERALL and c > 0.0:
            overall *= c
    return overall

# ==========================================
# MAIN EXECUTION
# ==========================================
def main():
    device_id = get_emptiest_gpu_safely()
    device = torch.device(f"cuda:{device_id}" if device_id != -1 else "cpu")
    
    print(f"DEBUG: Loading Config, Tokenizer, and XLM-R+CRF Model from {MODEL_DIR}...")
    if not os.path.exists(MODEL_DIR):
        print(f"❌ Error: {MODEL_DIR} not found.")
        return
        
    config = AutoConfig.from_pretrained(MODEL_DIR)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
    
    label_list = [config.id2label[k] if isinstance(k, int) else config.id2label[str(k)] 
                  for k in sorted([int(k) for k in config.id2label.keys()])]
                  
    model = BertCRFForTokenClassification(config, MODEL_DIR, label_list)
    
    weights_path = os.path.join(MODEL_DIR, "pytorch_model.bin")
    if os.path.exists(weights_path):
        model.load_state_dict(torch.load(weights_path, map_location=device))
    
    model.to(device)
    model.eval()

    all_fields = [
        "flat", "floor", "building_name", "block", "phase",
        "estate_name", "village_name", "building_number",
        "street_name", "sub_district", "district", "region"
    ]
    
    tag_map = {
        "flat": "UNIT", "floor": "FLOOR", "block": "BLOCK",
        "phase": "PHASE", "building_name": "BUILDING_NAME",
        "estate_name": "ESTATE_NAME", "village_name": "VILLAGE_NAME",
        "building_number": "BUILDING_NUMBER", "street_name": "STREET_NAME",
        "sub_district": "SUB_DISTRICT", "district": "DISTRICT", "region": "REGION",
    }
    
    # Tracking Metrics Across Multiple Thresholds
    total_samples = 0
    total_time = 0.0
    
    # Nested dictionary to store stats per threshold: stats[threshold][field]['TP']
    stats = {t: {f: {'TP': 0, 'FP': 0, 'FN': 0} for f in all_fields} for t in THRESHOLDS}
    exact_matches = {t: 0 for t in THRESHOLDS}

    print("🚀 Running batch evaluation...")
    with open(TEST_FILE, "r", encoding="utf-8") as file:
        raw_lines = [line.strip() for line in file if line.strip() and not line.startswith('#')]
    
    test_data = [json.loads(line) for line in raw_lines]
    
    with open(LOG_FILE, "w", encoding="utf-8") as log:
        for i in tqdm(range(0, len(test_data), BATCH_SIZE), desc="Batches"):
            batch = test_data[i:i + BATCH_SIZE]
            batch_texts = [item["input"].strip() for item in batch]
            
            start_time = time.perf_counter()
            
            encoded = tokenizer(
                batch_texts, padding=True, truncation=True,
                max_length=MAX_LEN, return_offsets_mapping=True, return_tensors="pt"
            )
            
            input_ids = encoded["input_ids"].to(device)
            attention_mask = encoded["attention_mask"].to(device)
            offsets_batch = encoded["offset_mapping"].cpu().numpy()
            
            with torch.no_grad():
                batch_predictions, batch_emissions = model(input_ids=input_ids, attention_mask=attention_mask)
                batch_probs = torch.softmax(batch_emissions, dim=-1).cpu()

            total_time += time.perf_counter() - start_time

            for b_idx, item in enumerate(batch):
                address = batch_texts[b_idx]
                ground_truth_flat = flatten_json(item.get("output", {}))
                
                prediction_ids = batch_predictions[b_idx]
                offsets = offsets_batch[b_idx]
                item_probs = batch_probs[b_idx]
                
                char_tags = ["O"] * len(address)
                char_confs = [0.0] * len(address)

                for idx, tag_id in enumerate(prediction_ids):
                    start, end = offsets[idx]
                    if start == end: continue 

                    tag = label_list[tag_id]
                    conf = item_probs[idx, tag_id].item()

                    for c in range(start, end):
                        if char_tags[c] == "O":
                            char_tags[c] = tag
                            char_confs[c] = conf
                            
                parsed_entities = []
                for match in re.finditer(r"[a-zA-Z0-9]+|[\u4e00-\u9fff]|[^\s]", address):
                    start_idx = match.start()
                    tag = char_tags[start_idx]
                    conf = char_confs[start_idx]
                    entity_group = tag.replace("B-", "").replace("I-", "") if tag != "O" else "O"
                    
                    parsed_entities.append({
                        "entity_group": entity_group, "word": match.group(), "conf": conf,
                    })

                extracted_data, conf_output = extract_3d_components(parsed_entities)
                predicted_flat = flatten_json(assemble_compact_json(extracted_data))
                overall_conf = compute_overall_confidence(conf_output)

                total_samples += 1
                
                # Evaluate against ALL thresholds
                for t in THRESHOLDS:
                    t_is_perfect_match = True
                    
                    for field in all_fields:
                        pred_val = predicted_flat.get(field, "")
                        gt_val = ground_truth_flat.get(field, "")
                        
                        tag = tag_map.get(field, field.upper())
                        field_conf = conf_output.get(tag, 0.0)
                        
                        # Apply Threshold: If confidence is lower than threshold, pretend model guessed nothing ("")
                        t_pred_val = pred_val if field_conf >= t else ""
                        
                        if t_pred_val != gt_val:
                            t_is_perfect_match = False
                            
                        # Calculate TP, FP, FN based on the thresholded prediction
                        if t_pred_val and gt_val:
                            if t_pred_val == gt_val:
                                stats[t][field]['TP'] += 1
                            else:
                                stats[t][field]['FP'] += 1
                                stats[t][field]['FN'] += 1
                        elif t_pred_val and not gt_val:
                            stats[t][field]['FP'] += 1
                        elif not t_pred_val and gt_val:
                            stats[t][field]['FN'] += 1
                    
                    if t_is_perfect_match: 
                        exact_matches[t] += 1

                # Write standard baseline logs (Threshold 0.0) to file
                base_perfect = (exact_matches[0.0] > 0) # Just checking if baseline was perfect
                log.write(f"Original: {address}\n")
                
                for field in all_fields:
                    pred_val = predicted_flat.get(field, "")
                    gt_val = ground_truth_flat.get(field, "")
                    if pred_val or gt_val:
                        status = "✅" if pred_val == gt_val else "❌"
                        tag = tag_map.get(field, field.upper())
                        conf_str = f"  conf={conf_output.get(tag, 0.0):.4f}" if tag in conf_output else ""
                        log.write(f" {status} {field.upper()}:{conf_str}\n")
                        log.write(f"   PRED: {pred_val if pred_val else '[None]'}\n")
                        log.write(f"   TRUE: {gt_val if gt_val else '[None]'}\n")

                log.write(f"Per-label confidences : { {k: round(v, 4) for k, v in conf_output.items()} }\n")
                log.write(f"Overall confidence    : {overall_conf:.6f}\n")
                log.write("-" * 50 + "\n")

    # --- FINAL REPORTS ---
    print("\n" + "=" * 65)
    print("📊 ADVANCED MULTI-THRESHOLD EVALUATION")
    print("=" * 65)
    print(f"Total Addresses Tested: {total_samples}")
    print(f"⏱️ Total Inference runtime: {total_time:.4f} seconds\n")

    for t in THRESHOLDS:
        title = "BASELINE (ALL PREDICTIONS)" if t == 0.0 else f"ONLY PREDICTIONS >= {int(t*100)}% CONFIDENCE"
        print("=" * 65)
        print(f"🚀 {title}")
        print("=" * 65)
        
        if total_samples > 0:
            exact_match_acc = (exact_matches[t] / total_samples) * 100
            print(f"Whole-Address Perfect Match : {exact_match_acc:.2f}% ({exact_matches[t]}/{total_samples})\n")
            
            print(f"{'FIELD':<16} | {'PRECISION':<9} | {'RECALL':<9} | {'F1-SCORE':<9} | {'SUPPORT'}")
            print("-" * 65)
            
            macro_f1 = 0
            valid_fields = 0
            
            for field in all_fields:
                tp = stats[t][field]['TP']
                fp = stats[t][field]['FP']
                fn = stats[t][field]['FN']
                
                support = tp + fn
                
                precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
                recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
                
                if support > 0:
                    macro_f1 += f1
                    valid_fields += 1
                    
                p_str = f"{precision*100:>5.1f}%"
                r_str = f"{recall*100:>5.1f}%"
                f1_str = f"{f1*100:>5.1f}%"
                
                print(f"{field:<16} | {p_str:<9} | {r_str:<9} | {f1_str:<9} | {support}")

            if valid_fields > 0:
                print("-" * 65)
                print(f"{'MACRO AVERAGE':<16} | {'-':<9} | {'-':<9} | {(macro_f1/valid_fields)*100:>5.1f}%  |")
        print("\n")

    print(f"✅ Processing complete. Raw baseline logs saved to {LOG_FILE}")

if __name__ == "__main__":
    main()